# 04 – Task (a): final regular-season ranking

**Business goal:** before the season starts, know how the teams will finish.
**DM goal:** regression of each team's **points% of season t** (`Pts / 2G`), then sort the teams (league, conference, division). Predicting a continuous score and sorting it works for any grouping and keeps ties rare.

| Metric | Why |
|---|---|
| **Spearman ρ** (main) | The client cares about the order, not the exact points. ρ compares the predicted and real ranks. |
| Kendall τ | Same idea, more robust with few teams; reported for completeness |
| Rank MAE | "On average we miss by X places" – easy to explain |
| Points% MAE | Error of the regression itself |
| Division Spearman | The official `rank` column is within the division |
| Playoff accuracy / ROC-AUC | Top 8 per conference by predicted points → do we pick the right playoff teams? |

**Success criterion:** beat the baseline "same as last season" in a statistically significant way.

In [ ]:
import sys, json, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50, "display.width", 200, "display.precision", 3)
from features import TEAM_FEATURES
from validation import (TEST_YEARS, MIN_YEAR, walk_forward, ranking_metrics, friedman_nemenyi,
                        plot_cd, wilcoxon_vs, FilterWrapperSelector)
from models import regressors
from experiment import run_regression, summary, _fit
from sklearn.base import clone

df = pd.read_csv("../data/processed/features.csv")
fl = json.load(open("../data/processed/feature_lists.json"))
FEATS, RK = fl["features"], fl["roster_known"]
hist = df[df.is_future == 0].reset_index(drop=True)
future = df[df.is_future == 1].reset_index(drop=True)
print(len(FEATS), "features |", hist.year.min(), "-", hist.year.max(), "| test seasons:", TEST_YEARS)

## 1. Experimental setup: walk-forward validation

To test season *T* we train on **every season before T** and predict *T*. This repeats for T = 2000 … 2009 (2004 does not exist) → **9 independent test seasons**, so every model gets 9 scores and we can run statistical tests.

- Random k-fold would train on 2008 to predict 2003: information from the future → over-optimistic.
- Hyper-parameters are tuned **inside** each training set, on an inner temporal split (validate on each of the last 3 training seasons).
- Imputation, winsorisation, scaling and feature selection are steps of the pipeline → fitted on training rows only.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.4))
for i, ty in enumerate(TEST_YEARS):
    ax.barh(i, ty - MIN_YEAR, left=MIN_YEAR, color="#1f4e79", height=.6)
    ax.barh(i, 1, left=ty, color="#e67e22", height=.6)
ax.set_yticks(range(len(TEST_YEARS)), [f"split {i+1}" for i in range(len(TEST_YEARS))])
ax.invert_yaxis(); ax.set_xlabel("season")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color="#1f4e79"), plt.Rectangle((0, 0), 1, 1, color="#e67e22")],
          labels=["train (all past seasons)", "test"], loc="lower left")
ax.set_title("Walk-forward: the model never sees the season it predicts, nor anything after it")
plt.show()

## 2. Baselines
- **Last season:** predicted points% = points% at t-1 (the ranking stays the same). Strong, because team quality is persistent (ρ ≈ 0.63 in the EDA).
- **3-season average:** smoother version.
- **League average:** everyone gets 0.5 → only the points MAE makes sense (no ranking).

In [ ]:
base_rows = []
for ty, tr, te in walk_forward(hist):
    preds = {"Baseline: last season": te.pts_pct_lag1.values,
             "Baseline: 3-season avg": te.pts_pct_ma3.values,
             "Baseline: league avg": np.full(len(te), 0.5)}
    for name, p in preds.items():
        m = ranking_metrics(te, p)
        if name == "Baseline: league avg":
            m.update(spearman=np.nan, kendall=np.nan, div_spearman=np.nan, playoff_auc=np.nan)
        base_rows.append({"model": name, "year": ty, **m})
base = pd.DataFrame(base_rows)
summary(base, ["spearman", "rank_mae", "pts_mae", "playoff_acc"])

## 3. Sampling for development: start small, then grow

Before running all algorithms we debug the pipeline on few seasons (fast) and check how much history helps. Ridge, no tuning.

In [ ]:
from sklearn.linear_model import Ridge
from imblearn.pipeline import Pipeline
from models import _prep
ridge_fixed = {"Ridge": (Pipeline(_prep() + [("m", Ridge(alpha=10))]), {})}
lc = []
for w in [3, 5, 10, 15, 20, None]:
    s, _, _ = run_regression(hist, FEATS, ridge_fixed, tune=False, train_window=w)
    lc.append({"train seasons": w or "all (21-30)", "spearman": s.spearman.mean(), "train_spearman": s.train_spearman.mean()})
lc = pd.DataFrame(lc)
ax = lc.plot(x="train seasons", y=["train_spearman", "spearman"], marker="o", figsize=(7, 3.5))
ax.set_title("Learning curve: more seasons -> less overfitting; gains flatten after ~10 seasons")
ax.set_ylabel("Spearman"); plt.show(); lc

## 4. Feature selection: filter + wrapper

`FilterWrapperSelector` (in `src/validation.py`): (1) **filter** – keep the 2k features with the highest mutual information with the target; (2) **wrapper** – RFE with a linear model keeps k of them. It is fitted inside each training set.

The curve below is an analysis (Ridge, fixed α). In the model comparison, k itself is a hyper-parameter chosen on the inner validation (`fs__k ∈ {8, 16, all}`).

In [ ]:
from sklearn.feature_selection import SelectKBest, mutual_info_regression
curve = []
for k in [1, 2, 4, 6, 8, 12, 16, 24, len(FEATS)]:
    for kind in ["filter (MI)", "filter + wrapper (MI -> RFE)"]:
        sel = (SelectKBest(mutual_info_regression, k=k) if kind.startswith("filter (")
               else FilterWrapperSelector(k=k, task="regression"))
        pipe = Pipeline(_prep() + [("fs", sel), ("m", Ridge(alpha=10))])
        s, _, _ = run_regression(hist, FEATS, {kind: (pipe, {})}, tune=False)
        curve.append({"k": k, "method": kind, "spearman": s.spearman.mean()})
curve = pd.DataFrame(curve)
ax = sns.lineplot(data=curve, x="k", y="spearman", hue="method", marker="o")
ax.axhline(base[base.model == "Baseline: last season"].spearman.mean(), ls="--", color="grey", label="baseline")
ax.set_title("A handful of features is enough; adding all 44 adds noise"); ax.legend(); plt.show()
curve.pivot(index="k", columns="method", values="spearman")

In [ ]:
# Which features does the selector pick most often (k = 8, over the 9 training sets)?
picked = []
for ty, tr, te in walk_forward(hist):
    pre = Pipeline(_prep()).fit(tr[FEATS])
    fs = FilterWrapperSelector(k=8).fit(pre.transform(tr[FEATS]), tr.pts_pct)
    picked += list(np.array(FEATS)[fs.support_])
pd.Series(picked).value_counts().to_frame("times selected (out of 9)").head(15)

## 5. Model comparison (8 algorithms with different language bias)

| Algorithm | Bias / how it works | Main parameters tuned |
|---|---|---|
| Ridge | linear, L2 penalty shrinks all coefficients | `alpha` (more = simpler) |
| Lasso | linear, L1 penalty sets coefficients to 0 (selection) | `alpha` |
| kNN | the average of the k most similar past teams | `k` (more = smoother), weights |
| SVR | linear/RBF function, ignores errors smaller than ε | `C` (less = smoother), `ε`, `γ` |
| Decision tree | axis-parallel rules, piecewise constant | `max_depth`, `min_samples_leaf` |
| Random Forest | average of many decorrelated trees (bagging) | depth, leaf size, `max_features` |
| Gradient Boosting | trees added one by one to fix the errors so far | learning rate, depth |
| MLP | neural network, non-linear combinations | hidden layers, `alpha` (L2) |

Each one is tuned with grid search on the inner temporal split (scoring: points MAE), and k of the feature selector is tuned too.

In [ ]:
import time
t0 = time.time()
scores, preds, params = run_regression(hist, FEATS, regressors())
print(f"{time.time()-t0:.0f}s")
allscores = pd.concat([scores, base])
import os
os.makedirs("../data/processed/results", exist_ok=True)
allscores.to_csv("../data/processed/results/task_a_scores.csv", index=False)
preds.to_csv("../data/processed/results/task_a_predictions.csv", index=False)
summary(allscores, ["spearman", "kendall", "rank_mae", "pts_mae", "div_spearman", "playoff_acc", "playoff_auc"])

In [ ]:
order = allscores.groupby("model").spearman.mean().sort_values(ascending=False).dropna().index
fig, ax = plt.subplots(figsize=(11, 4))
sns.boxplot(data=allscores[allscores.model.isin(order)], x="model", y="spearman", order=order, ax=ax, color="#9ecae1")
sns.stripplot(data=allscores[allscores.model.isin(order)], x="model", y="spearman", order=order, ax=ax, color="#1f4e79", size=4)
ax.tick_params(axis="x", rotation=30); ax.set_title("Spearman per test season: the spread between seasons is larger than between models")
plt.show()

In [ ]:
# chosen hyper-parameters (most frequent over the 9 splits)
params.drop(columns="year").groupby("model").agg(lambda s: s.dropna().astype(str).mode().iloc[0] if s.notna().any() else "-")

## 6. Are the differences significant?

- **Friedman test:** do the models have the same average rank over the 9 test seasons? (each season is a block)
- **Nemenyi post-hoc:** which pairs differ? Two models whose average ranks differ by less than the critical difference (CD) are not significantly different.
- **Wilcoxon signed-rank:** best model vs. the baseline, paired by season.

In [ ]:
piv = allscores.pivot(index="year", columns="model", values="spearman").drop(columns="Baseline: league avg")
stat, p, ranks, cd = friedman_nemenyi(piv)
print(f"Friedman chi2 = {stat:.2f}, p = {p:.4f}")
fig, ax = plt.subplots(figsize=(9, 4)); plot_cd(ranks, cd, ax); plt.show()
best = piv.drop(columns=[c for c in piv if c.startswith("Baseline")]).mean().idxmax()
w = wilcoxon_vs(piv, best, "Baseline: last season")
print(f"Best model: {best}  mean rho = {piv[best].mean():.3f} vs baseline {piv['Baseline: last season'].mean():.3f}")
print(f"Wilcoxon {best} vs baseline: statistic = {w.statistic:.1f}, p = {w.pvalue:.3f}")
print("Seasons where the best model beats the baseline:", int((piv[best] > piv["Baseline: last season"]).sum()), "of", len(piv))
# The same test on the points error (lower is better)
pm = allscores.pivot(index="year", columns="model", values="pts_mae")
w2 = wilcoxon_vs(pm, best, "Baseline: last season")
print(f"Points% MAE: {best} {pm[best].mean():.4f} vs baseline {pm['Baseline: last season'].mean():.4f}; "
      f"Wilcoxon p = {w2.pvalue:.4f}; better in {int((pm[best] < pm['Baseline: last season']).sum())} of {len(pm)} seasons")

## 7. Overfitting

`train_spearman` is measured on the last training season (in-sample), `spearman` on the test season.

In [ ]:
ov = scores.groupby("model")[["train_spearman", "spearman"]].mean()
ov["gap"] = ov.train_spearman - ov.spearman
ov.sort_values("gap", ascending=False)

In [ ]:
# Validation curves: one parameter at a time, everything else fixed, walk-forward test score vs. training score
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
def vcurve(make_model, values, scale):
    out = []
    for v in values:
        pipe = Pipeline(_prep(scale=scale) + [("m", make_model(v))])
        s, _, _ = run_regression(hist, FEATS, {"x": (pipe, {})}, tune=False)
        out.append({"param": v, "train": s.train_spearman.mean(), "test": s.spearman.mean()})
    return pd.DataFrame(out)
vt = vcurve(lambda d: DecisionTreeRegressor(max_depth=d, random_state=0), [1, 2, 3, 4, 6, 8, 12, None], False)
vk = vcurve(lambda k: KNeighborsRegressor(n_neighbors=k), [1, 3, 5, 10, 25, 50, 100, 200], True)
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
vt.assign(param=vt.param.astype(str)).plot(x="param", y=["train", "test"], marker="o", ax=axes[0])
axes[0].set(xlabel="max_depth", ylabel="Spearman", title="Tree: deeper trees memorise the training seasons")
vk.plot(x="param", y=["train", "test"], marker="o", ax=axes[1], logx=True)
axes[1].set(xlabel="k (neighbours)", title="kNN: k=1 copies one past team; large k smooths towards the mean")
plt.show()

## 8. Model improvement log

Each row is one iteration: what we changed and its effect on the mean test Spearman. A change is kept only if it helps.

In [ ]:
ridge_t = {"Ridge": regressors(select=False)["Ridge"]}
def mean_rho(feats, models=ridge_t, data=hist):
    s, _, _ = run_regression(data, feats, models)
    return s.spearman.mean()
rk_df = hist.dropna(subset=["rk_prev_pts"])
log = [
    ("Baseline: same ranking as last season", base[base.model == "Baseline: last season"].spearman.mean()),
    ("Ridge on points% t-1 only", mean_rho(["pts_pct_lag1"])),
    ("+ goal-based features (gd 3-season avg, luck)", mean_rho(["pts_pct_lag1", "gd_pg_ma3", "luck_lag1"])),
    ("+ all team features", mean_rho([f for f in FEATS if f in TEAM_FEATURES])),
    ("+ roster, goalie and coach features (all 44)", mean_rho(FEATS)),
    ("+ filter/wrapper feature selection (k tuned)", mean_rho(FEATS, {"Ridge": regressors()["Ridge"]})),
    (f"best algorithm ({best})", piv[best].mean()),
    ("+ roster of season t known (rk_* features, Ridge + selection)", mean_rho(FEATS + RK, {"Ridge": regressors()["Ridge"]})),
]
log = pd.DataFrame(log, columns=["change", "mean Spearman"])
log["delta"] = log["mean Spearman"].diff()
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(range(len(log)), log["mean Spearman"], marker="o", color="#1f4e79")
ax.set_xticks(range(len(log)), [f"#{i}" for i in range(len(log))]); ax.set_ylabel("mean test Spearman")
ax.set_title("Iterations: selecting few good features helps; knowing the new roster helps most")
plt.show(); log

## 9. Interpretation

### 9.1 Permutation importance (best model, on the test seasons)
For each test season we shuffle one feature and measure how much the error grows. Computed on **test** data, so it shows what the model really uses to generalise.

In [ ]:
from sklearn.inspection import permutation_importance
pipe_best, grid_best = regressors()[best]
imps = []
for ty, tr, te in walk_forward(hist):
    est = _fit(pipe_best, grid_best, tr[FEATS], tr.pts_pct, tr.year, "neg_mean_absolute_error")
    r = permutation_importance(est, te[FEATS], te.pts_pct, scoring="neg_mean_absolute_error", n_repeats=10, random_state=0)
    imps.append(pd.Series(r.importances_mean, index=FEATS))
imp = pd.concat(imps, axis=1).mean(axis=1).sort_values(ascending=False)
ax = (imp.head(12) * 1000).iloc[::-1].plot.barh(figsize=(8, 4.5), color="#1f4e79")
ax.set(xlabel="increase in points% MAE when shuffled (x 1000)", title=f"{best}: what drives the predicted ranking")
plt.show()

### 9.2 White-box models: Ridge coefficients and a shallow tree
Fitted on all seasons 1979–2009 (standardised features, so the coefficients are comparable).

In [ ]:
pre = Pipeline(_prep()).fit(hist[FEATS])
Xs = pre.transform(hist[FEATS])
sel = FilterWrapperSelector(k=8).fit(Xs, hist.pts_pct)
r = Ridge(alpha=10).fit(Xs[:, sel.support_], hist.pts_pct)
coef = pd.Series(r.coef_, index=np.array(FEATS)[sel.support_]).sort_values()
ax = coef.plot.barh(figsize=(7, 3.5), color=np.where(coef > 0, "#1f4e79", "#c0392b"))
ax.set(xlabel="effect on points% of +1 std", title="Ridge (8 selected features)"); plt.show()
coef.to_frame("coefficient (per std)")

In [ ]:
from sklearn.tree import DecisionTreeRegressor, export_text, plot_tree
from sklearn.impute import SimpleImputer
tree_feats = ["pts_pct_lag1", "luck_lag1", "gd_pg_ma3", "avg_age_lag1", "team_sv_pct_lag1", "new_coach", "is_new_team"]
Xt = SimpleImputer(strategy="median").fit_transform(hist[tree_feats])
tree = DecisionTreeRegressor(max_depth=3, min_samples_leaf=30, random_state=0).fit(Xt, hist.pts_pct)
fig, ax = plt.subplots(figsize=(18, 6))
plot_tree(tree, feature_names=tree_feats, filled=True, rounded=True, precision=3, fontsize=9, ax=ax); plt.show()
print(export_text(tree, feature_names=tree_feats, decimals=3))

## 10. Final predictions: season 2010

Retrain the best configuration on **all** seasons 1979–2009 (tuned on the last 3 seasons, as before) and predict the 2010 points% of each team. The predicted ranking is reported per conference and division; the top 8 of each conference are the predicted playoff teams.

(The `rk_*` roster features cannot be used for 2010: the 2010 rosters are not in the data.)

In [ ]:
final = _fit(*regressors()[best], hist[FEATS], hist.pts_pct, hist.year, "neg_mean_absolute_error")
future["pred_pts_pct"] = final.predict(future[FEATS])
future["pred_league_rank"] = future.pred_pts_pct.rank(ascending=False).astype(int)
future["pred_conf_rank"] = future.groupby("confID").pred_pts_pct.rank(ascending=False).astype(int)
future["pred_div_rank"] = future.groupby("divID").pred_pts_pct.rank(ascending=False).astype(int)
future["pred_playoffs"] = np.where(future.pred_conf_rank <= 8, "yes", "no")
future["pred_points_82g"] = (future.pred_pts_pct * 164).round(0).astype(int)
out = future.sort_values(["confID", "pred_conf_rank"])[["confID", "divID", "name", "pts_pct_lag1", "pred_pts_pct", "pred_points_82g",
                                                       "pred_conf_rank", "pred_div_rank", "pred_league_rank", "pred_playoffs"]]
out.to_csv("../data/processed/results/task_a_predictions_2010.csv", index=False)
print("best params:", final.best_params_)
out

## Conclusions and reflection

- **Ranking:** the best model (Ridge with 8 selected features) reaches a mean Spearman of **≈ 0.55**, the same as the baseline "same ranking as last season" (Wilcoxon p ≈ 0.8). Friedman says the models are not all equal (p ≈ 0.04), but the Nemenyi critical difference shows that the linear models, SVR, MLP, RF, kNN and the baseline are not significantly different; only the single tree and gradient boosting are clearly worse.
- **Points:** the model's points% error is **significantly lower** than the baseline's (0.059 vs 0.067, better in every test season). The model learned **regression to the mean**: good teams are predicted a bit worse and bad teams a bit better (Ridge coefficient of points% t-1 < 1, negative coefficient of `made_playoffs_lag1` once points are in). This is the "luck reverts" finding of notebook 02 at work.
- **Why the ranking is hard:** since 2000 the NHL has 30 teams, a salary cap (2005) and much more parity; injuries and trades during the season are not in the data. With only last season's information, about 55% of the order is predictable.
- **What helps:** few, well-chosen features (points% t-1 and t-2, goal differential over 3 seasons, luck, goaltending, coach record). Knowing the **new roster** (`rk_*`) is the largest single gain (ρ 0.55 → 0.57–0.60) → ask the client whether it may be used.
- **What does not help:** using all 44 features (ρ drops to 0.52), deep trees and kNN with small k (they memorise the training seasons, see the validation curves and the train–test gap).
- **Next iteration:** confirm the roster question; tune directly for Spearman instead of MAE; weight recent seasons more.